# 03. Hybrid Recommendation
02의 실제 산출물(v2) 또는 보완된 v3를 읽어 추천 엔진을 구현합니다. 평가/최적화는 다음 04 단계입니다. v2의 S는 최소 200건 기준으로 모두 탈락하므로 R+C로 동작합니다. 가중치 0.4/0.3/0.3은 실험 초기값입니다.

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
from google.colab import drive
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/2026_02_ROKEY/07_cobot2/recommender')
VERSION='pattern_analysis_v3' # 02-1 완료 후 v3로 변경
SRC=BASE/VERSION
OUT=BASE/('hybrid_'+VERSION)
OUT.mkdir(parents=True,exist_ok=True)


Mounted at /content/drive


## 1. 입력 확인
R 행렬의 행은 user_ids, 열은 metadata.item_ids 순서입니다. C는 aisle ID, S는 상품 ID로 조회합니다.

In [3]:
meta=json.loads((SRC/'pattern_metadata.json').read_text())
ids=meta['item_ids']; idx={x:i for i,x in enumerate(ids)}
aisle=meta['item_aisle']; names=pd.read_csv(SRC/'mapping_coverage.csv').set_index('id')['name'].to_dict()
R=sparse.load_npz(SRC/'repurchase_R.npz').tocsr()
users=np.load(SRC/'user_ids.npy',allow_pickle=False)
user_index={int(u):i for i,u in enumerate(users)}
S=pd.read_csv(SRC/'same_aisle_scores.csv'); C=pd.read_csv(SRC/'cross_aisle_scores.csv')
assert R.shape==(len(users),len(ids)) and len(user_index)==len(users)
assert set(ids)==set(aisle)==set(names)
assert np.isfinite(R.data).all() and ((R.data>=0)&(R.data<=1)).all()
print('R shape:',R.shape,'S pairs:',len(S),'C pairs:',len(C))


R shape: (206209, 27) S pairs: 30 C pairs: 1386


## 2. 연관 점수 구성
Confidence는 이미 0~1입니다. Lift는 max(0, 1−1/Lift)로 변환하여 양의 연관성만 0~1에 담습니다. 둘의 평균을 사용합니다(검증 전 설정). 최소 동시 구매 200건 미만은 근거 부족으로 표시하며, 실제 0점과 구분합니다. S는 한 코너만 담은 주문도 포함하는 1~5 코너, C는 2~5 코너를 기본으로 사용합니다.

In [4]:
MIN_COUNT=200
S_SCOPE='aisles_1_5'; C_SCOPE='aisles_2_5'
LIFT_WEIGHT=.5
WEIGHTS=(.4,.3,.3)
def pair_scores(df,scope):
    t=df[df.scope.eq(scope)].copy()
    assert not t.duplicated(['source','target']).any()
    t['valid']=t.observed & t['count'].ge(MIN_COUNT)
    t['lift_scaled']=1-1/t.lift.clip(lower=1)
    t['score']=(1-LIFT_WEIGHT)*t.confidence+LIFT_WEIGHT*t.lift_scaled
    return {(r.source,r.target):float(r.score) for r in t.itertuples() if r.valid}
s_pairs=pair_scores(S,S_SCOPE); c_pairs=pair_scores(C,C_SCOPE)
# 상품군별 인기도: R * 고객별 주문 수를 합산 / 전체 주문 수
counts=np.load(SRC/'user_order_counts.npy',allow_pickle=False)
assert len(counts)==len(users) and counts.sum()>0
pop=np.asarray(R.T @ counts).ravel()/counts.sum()
print('유효 S:',len(s_pairs),'유효 C:',len(c_pairs))
print('인기순:',[(names[ids[i]],round(float(pop[i]),3)) for i in np.argsort(-pop)[:5]])


유효 S: 8 유효 C: 434
인기순: [('소형 우유팩', 0.244), ('요구르트', 0.244), ('생수', 0.191), ('식빵', 0.164), ('계란 모형', 0.137)]


## 3. 추천 엔진
장바구니 전체의 관계를 평균합니다. 후보별 가중치를 임의로 바꾸지 않고, 현재 요청에서 어떤 분석 채널도 유효한 관계를 만들지 못할 때만 해당 채널을 전체 후보에서 비활성화합니다. 후보별 근거 부족은 0 기여로 처리하되 유효 관계 수를 별도 표시합니다. 이 정책 자체도 04에서 비교할 대상입니다.

In [5]:
def recommend(basket=(), user_id=None, stock=None, weights=WEIGHTS, k=3, history_scores=None):
    basket=list(dict.fromkeys(basket))
    if any(x not in idx for x in basket): raise ValueError('미등록 상품 ID: '+str(set(basket)-set(ids)))
    w=np.asarray(weights,dtype=float)
    if w.shape!=(3,) or not np.isfinite(w).all() or (w<0).any() or w.sum()==0: raise ValueError('잘못된 가중치')
    if history_scores is not None:
        r=np.array([history_scores.get(x,0) for x in ids],dtype=float)
        if not np.isfinite(r).all() or ((r<0)|(r>1)).any(): raise ValueError('R은 0~1')
        has_r=bool(history_scores)
    else:
        has_r=user_id in user_index
        r=R.getrow(user_index[user_id]).toarray().ravel() if has_r else np.zeros(len(ids))
    rows=[]
    for j in ids:
        if j in basket or (stock is not None and stock.get(j,0)<=0): continue
        sv=[s_pairs[(i,j)] for i in basket if aisle[i]==aisle[j] and (i,j) in s_pairs]
        cv=[c_pairs[(aisle[i],aisle[j])] for i in basket if aisle[i]!=aisle[j] and (aisle[i],aisle[j]) in c_pairs]
        rows.append(dict(id=j,name=names[j],R=float(r[idx[j]]),S=sum(sv)/max(1,len(basket)),C=sum(cv)/max(1,len(basket)),S_evidence=len(sv),C_evidence=len(cv),popularity=float(pop[idx[j]])))
    columns=['id','name','R','S','C','S_evidence','C_evidence','popularity','score','reason']
    if not rows: return pd.DataFrame(columns=columns)
    t=pd.DataFrame(rows)
    active=np.array([has_r,t.S_evidence.gt(0).any(),t.C_evidence.gt(0).any()])
    effective=w*active
    if effective.sum()>0:
        effective/=effective.sum(); t['score']=t[['R','S','C']].to_numpy()@effective
        t['reason']=np.where(t.score.gt(0),'hybrid','popularity_fallback')
    else:
        t['score']=t.popularity; t['reason']='popularity_fallback'
    t=t.sort_values(['score','popularity','id'],ascending=[False,False,True]).head(k).reset_index(drop=True)
    t.attrs['effective_weights']=effective.tolist()
    return t


## 4. 실제 데이터 기반 테스트
아래는 추천 출력 확인이지 정확도 평가가 아닙니다. 신규 고객, 기존 고객, 품절, 빈 장바구니를 비교합니다.

In [6]:
cases=[('신규 / 참치',['tuna_can'],None,None),('신규 / 과자',['snack_bag'],None,None),('기존 / 과자',['snack_bag'],int(users[0]),None),('기존 / 빈 장바구니',[],int(users[0]),None),('신규 / 빈 장바구니',[],None,None),('품절 테스트',['snack_bag'],None,{x:int(x!='cookie') for x in ids})]
results=[]
for label,basket,u,stock in cases:
    t=recommend(basket,u,stock)
    print(label,'effective R/S/C:',t.attrs.get('effective_weights')); display(t.round(4))
    assert not set(t.id)&set(basket)
    if stock is not None: assert all(stock.get(x,0)>0 for x in t.id)
    results.append(t.assign(case=label))
assert recommend([],stock={}).empty
pd.concat(results,ignore_index=True).to_csv(OUT/'recommendation_examples.csv',index=False)


신규 / 참치 effective R/S/C: [0.0, 0.0, 1.0]


,id,name,R,S,C,S_evidence,C_evidence,popularity,score,reason
0,sausage,소시지,0.0,0.0,0.0833,0,1,0.0841,0.0833,hybrid
1,egg_pack,계란 모형,0.0,0.0,0.0678,0,1,0.1370,0.0678,hybrid
2,water,생수,0.0,0.0,0.0523,0,1,0.1910,0.0523,hybrid


신규 / 과자 effective R/S/C: [0.0, 0.5, 0.5]


,id,name,R,S,C,S_evidence,C_evidence,popularity,score,reason
0,tortilla_chips,나초·토르티야 칩,0.0,0.4038,0.0000,1,0,0.0473,0.2019,hybrid
1,cookie,쿠키,0.0,0.0000,0.2546,0,1,0.0590,0.1273,hybrid
2,choco_box,초콜릿 상자,0.0,0.0000,0.2182,0,1,0.0465,0.1091,hybrid


기존 / 과자 effective R/S/C: [0.4, 0.3, 0.3]


,id,name,R,S,C,S_evidence,C_evidence,popularity,score,reason
0,cola,콜라,1.0,0.0,0.1200,0,1,0.0849,0.4360,hybrid
1,nuts,견과류,0.9,0.0,0.0897,0,1,0.0807,0.3869,hybrid
2,cereal,시리얼,0.3,0.0,0.0186,0,1,0.0925,0.1256,hybrid


... (2 outputs left out)

신규 / 빈 장바구니 effective R/S/C: [0.0, 0.0, 0.0]


,id,name,R,S,C,S_evidence,C_evidence,popularity,score,reason
0,milk_pack,소형 우유팩,0.0,0.0,0.0,0,0,0.2445,0.2445,popularity_fallback
1,yogurt,요구르트,0.0,0.0,0.0,0,0,0.2438,0.2438,popularity_fallback
2,water,생수,0.0,0.0,0.0,0,0,0.1910,0.1910,popularity_fallback


품절 테스트 effective R/S/C: [0.0, 0.5, 0.5]


,id,name,R,S,C,S_evidence,C_evidence,popularity,score,reason
0,tortilla_chips,나초·토르티야 칩,0.0,0.4038,0.0000,1,0,0.0473,0.2019,hybrid
1,choco_box,초콜릿 상자,0.0,0.0000,0.2182,0,1,0.0465,0.1091,hybrid
2,dark_chocolate,다크초콜릿,0.0,0.0000,0.2182,0,1,0.0283,0.1091,hybrid


## 5. 웹 연동용 점수 저장
개인별 Instacart ID와 이력은 웹에 내보내지 않습니다. 웹은 자체 고객 이력으로 R을 계산합니다. 아직 최적화되지 않은 설정이며 기존 recommender.py는 변경하지 않습니다.

In [7]:
payload={'schema_version':1,'source':VERSION,'item_ids':ids,'item_aisle':aisle,'names':names,'popularity':dict(zip(ids,map(float,pop))), 'weights':dict(zip(['R','S','C'],WEIGHTS)), 'min_count':MIN_COUNT,'s_scope':S_SCOPE,'c_scope':C_SCOPE,'lift_weight':LIFT_WEIGHT,'s_pairs':[{'source':str(a),'target':str(b),'score':v} for (a,b),v in s_pairs.items()], 'c_pairs':[{'source':int(a),'target':int(b),'score':v} for (a,b),v in c_pairs.items()], 'validated':False}
(OUT/'hybrid_scores.json').write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf-8')
print('저장 완료:',OUT)


저장 완료: /content/drive/MyDrive/2026_02_ROKEY/07_cobot2/recommender/hybrid_pattern_analysis_v3


## 다음: 04 Evaluation & Optimization
동일 매핑으로 마지막 주문을 변환하고 고객 단위로 validation/test를 분리합니다. 같은 후보와 숨긴 상품으로 인기순, R, S, C, S+C, R+C, R+S+C를 비교합니다. v2에서는 S 비활성화가 정상입니다. 66개 가중치 조합을 validation에서 탐색하고 test는 최종 1회 평가합니다. 신규 고객/기존 고객과 매핑 가능 주문의 비율을 따로 보고합니다.